# Wave 2 — Composite Scoring & Explainability
**Satu pipeline end-to-end:** raw dataset (Wave 0) → skor Module A, B, C (Wave 1) → normalisasi → composite score → penjelasan per perusahaan → `output_scores.json`.

Cukup **Runtime → Run all**. Tidak ada langkah manual di tengah.

### Keputusan desain (untuk dokumentasi & presentasi)
| Checklist | Keputusan | Alasan |
|---|---|---|
| Normalisasi 0–1 | **Berbasis threshold Wave 1**, bukan min-max. Skor mentah = 0 → 0.0, tepat di threshold → **0.5**, ≥ batas atas → 1.0. Modul yang *tidak* flag dibatasi maks 0.49, modul yang flag minimal 0.5. | Min-max bergantung pada isi dataset (satu outlier bikin semua skor lain mengecil, dan skor berubah tiap data baru). Dengan threshold, **≥ 0.5 selalu berarti "melewati batas flag"** di modul mana pun, jadi mudah dijelaskan ke pemeriksa. |
| Composite | **Rata-rata equal-weight** dari modul yang tersedia. | Sesuai checklist; tidak ada modul yang dianggap lebih penting tanpa dasar data. |
| Ranking & band | Band ditentukan oleh **jumlah modul yang flag** dan **kekuatan bukti terkuat** (skor modul tertinggi). Di dalam band, urutan = jumlah flag → skor modul tertinggi → composite. | Rata-rata bisa "mengencerkan" perusahaan yang curang di satu cara saja (misal hanya tidak setor), dan nilainya bias terhadap jumlah modul yang punya data. Band + skor terkuat membuat urutan adil terlepas dari kelengkapan data. Composite (rata-rata) tetap dihitung & ditampilkan. |
| INSUFFICIENT_DATA | Composite **tetap dihitung dari modul yang tersedia** (misal B+C), dengan kolom `coverage` (mis. 2/3) dan catatan di penjelasan. Kalau **semua** modul tidak bisa dinilai → band `Belum bisa dinilai`, composite `null`. | Perusahaan baru tetap bisa ketahuan kalau tidak setor iuran, tapi tim pemeriksa tahu bahwa penilaiannya belum lengkap (requirement #1: status netral, bukan aman). |

### File yang dibutuhkan (dari folder `Dummy Healthkathon`)
`headcount_timeseries`, `resign_records` → Module A · `payroll_timeseries`, `employer_master` → Module B · `remittance_timeseries` → Module C · `ground_truth` → evaluasi (opsional).

**Semua logika modul identik dengan notebook Wave 1 (A, B, C).** Sel cross-check (bagian 10) memastikan otomatis bahwa hasil di sini sama dengan file `module_*_scores.csv` dari Wave 1. Kalau ada yang beda, penyebabnya hampir pasti konfigurasi yang belum disamakan.


In [ ]:
import os, glob, json
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 220)
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running di Colab:", IN_COLAB)

## 1. Lokasi data & output (Google Drive)

In [ ]:
BASE_DIR  = "/content/drive/MyDrive/Healthkathon Engine"
DRIVE_DIR = f"{BASE_DIR}/Dummy Healthkathon"
OUT_DRIVE = f"{BASE_DIR}/wave 2"

if os.environ.get("HK_DATA_DIR"):
    DATA_DIR = Path(os.environ["HK_DATA_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
    if not (DATA_DIR / "employer_master.csv").exists():
        hits = glob.glob("/content/drive/MyDrive/**/employer_master.csv", recursive=True)
        if hits:
            DATA_DIR = Path(hits[0]).parent
            print("⚠ DRIVE_DIR tidak ditemukan, pakai:", DATA_DIR)
else:
    local_candidates = [Path("data/dummy"), Path("../data/dummy"), Path("data")]
    DATA_DIR = next((path for path in local_candidates if (path / "employer_master.csv").exists()), local_candidates[0])

OUT_DIR = Path(os.environ.get("HK_OUT_DIR", OUT_DRIVE if IN_COLAB else "output_wave2"))

print("DATA_DIR =", DATA_DIR); print("OUT_DIR  =", OUT_DIR)
for f in sorted(DATA_DIR.glob("*.csv")):
    print("  ✓", f.name)

## 2. Konfigurasi
Semua angka tuning dari Wave 1 dikumpulkan di sini. **Kalau kalian sudah dapat angka final dari sweep notebook A/B, tulis di sini.**

In [ ]:
# ---- Mapping kolom (None = auto-detect) -------------------------------------
COLUMN_MAP = {
    "hc_employer": None, "hc_period": None, "hc_count": None,
    "rs_employer": None, "rs_period": None, "rs_count": None,
    "pr_employer": None, "pr_period": None,
    "pr_avg_wage": "rata2_DPI", "pr_total_wage": None, "pr_single_wage": None,
    "pr_headcount": None, "pr_umr": None,
    "em_employer": None, "em_sector": None, "em_region": None, "em_scale": None, "em_umr": None,
    "rm_employer": None, "rm_period": None, "rm_expected": None, "rm_actual": None,
    "gt_employer": None, "gt_label": None,
}

# ---- UMP 2025 per provinsi (CEK ULANG ke sumber resmi) -----------------------
UMR_TABLE = {
    "DKI Jakarta": 5_396_761, "Jawa Barat": 2_191_238, "Jawa Tengah": 2_169_349,
    "Jawa Timur": 2_305_985, "DI Yogyakarta": 2_264_080, "Banten": 2_905_119,
    "Bali": 2_996_560, "Sumatera Utara": 2_992_559, "Sumatera Barat": 2_994_193,
    "Sumatera Selatan": 3_681_570, "Riau": 3_508_776, "Kepulauan Riau": 3_623_653,
    "Kalimantan Timur": 3_579_313, "Sulawesi Selatan": 3_657_527, "Papua": 4_285_848,
}

@dataclass(frozen=True)
class ConfigA:                     # Registration volatility
    Z_THRESHOLD: float = 2.0
    ROLLING_WINDOW: int = 6
    MIN_PERIODS: int = 3
    MIN_DROP_PCT: float = 0.10
    STD_FLOOR_ABS: float = 1.0
    STD_FLOOR_REL: float = 0.02
    CAP_MULT: float = 3.0          # skor mentah >= Z_THRESHOLD*CAP_MULT -> norm 1.0

@dataclass(frozen=True)
class ConfigB:                     # Peer-group wage benchmarking
    Z_THRESHOLD: float = 1.5
    MIN_PERSIST: float = 0.5
    MIN_COHORT_SIZE: int = 5
    MIN_PERIODS: int = 3
    UMR_PREFILTER_MULT: float | None = 1.5
    IQR_FLOOR: float = 0.05
    CAP_MULT: float = 3.0

@dataclass(frozen=True)
class ConfigC:                     # Contribution reconciliation
    TOL_PCT: float = 0.02          # selisih <= 2% dari iuran seharusnya dianggap pembulatan
    TOL_ABS: float = 10_000        # selisih <= Rp10.000 diabaikan (harus lewat KEDUANYA baru dihitung)
    MIN_CONSECUTIVE: int = 2       # flag hanya kalau kurang setor >= 2 bulan BERTURUT-TURUT
    ALLOW_CATCHUP: bool = True     # kurang bulan ini tapi dilunasi bulan depan = telat wajar, tidak dihitung
    MIN_PERIODS: int = 2           # < 2 bulan data setoran -> INSUFFICIENT_DATA
    CAP_PCT: float = 0.25          # kekurangan >= 25% -> skor ternormalisasi 1.0
    # dipakai HANYA kalau remittance tidak punya kolom "iuran seharusnya":
    CONTRIB_RATE: float = 0.05     # iuran BPJS Kesehatan PPU 5% (4% pemberi kerja + 1% pekerja)
    WAGE_CAP: float = 12_000_000   # batas atas upah perhitungan iuran

@dataclass(frozen=True)
class ConfigComposite:
    METHOD: str = "mean"           # "mean" (equal-weight, sesuai checklist) atau "max"
    STRONG_SCORE: float = 0.8      # 1 modul flag dengan skor >= ini -> band Tinggi

CFG_A, CFG_B, CFG_C, CFG_X = ConfigA(), ConfigB(), ConfigC(), ConfigComposite()

# Label ground truth per modul (dicek "mengandung", tidak case-sensitive)
POSITIVE_LABELS = {
    "A": ["PDUK", "HIDDEN", "HEADCOUNT", "EMPLOYEE", "KARYAWAN", "SEMBUNYI"],
    "B": ["WAGE", "UPAH", "GAJI", "SALARY", "DPI", "UNDERPAY"],
    "C": ["REMITTANCE", "SETOR", "IURAN", "CONTRIBUTION"],
}
NEGATIVE_LABELS = ["CLEAN", "NONE", "NORMAL", "-", "NAN"]

MODULE_LABEL = {"A": "Sembunyiin karyawan", "B": "Lapor gaji lebih rendah", "C": "Setoran iuran tidak sesuai"}

## 3. Load data + deteksi kolom

In [ ]:
FORBIDDEN_COLS = {"nama", "name", "nama_karyawan", "employee_name", "nik", "no_kartu",
                  "no_ktp", "ktp", "alamat", "address", "tanggal_lahir", "birth_date"}

CANDIDATES = {
    "employer": ["employer_id", "id_employer", "company_id", "perusahaan_id", "id_perusahaan",
                 "badan_usaha_id", "kode_badan_usaha", "kode_bu", "npp", "employer"],
    "period":   ["period", "periode", "month", "bulan", "year_month", "yearmonth", "date", "tanggal",
                 "resign_date", "tanggal_resign", "effective_date", "exit_date", "event_date"],
    "headcount": ["headcount", "active_headcount", "n_active", "active_employees", "jumlah_karyawan",
                  "jumlah_peserta", "n_employees", "employee_count", "registered_headcount", "hc", "n_peserta"],
    "rs_count": ["n_resign", "resign_count", "jumlah_resign", "n_exit", "exit_count", "count",
                 "n_keluar", "jumlah_keluar", "n_terminated", "jumlah"],
    "avg_wage": ["rata2_dpi", "rata_rata_dpi", "avg_dpi", "dpi", "avg_wage", "average_wage", "mean_wage",
                 "avg_reported_wage", "reported_avg_wage", "avg_salary", "rata_rata_upah", "upah_rata_rata",
                 "wage_per_employee", "avg_upah", "avg_gaji"],
    "total_wage": ["total_wage", "reported_wage_total", "total_reported_wage", "total_upah", "total_gaji",
                   "payroll_total", "total_payroll", "wage_total", "total_salary", "wage_bill", "total_dpi"],
    "single_wage": ["reported_wage", "wage", "upah", "gaji", "salary", "upah_dilaporkan"],
    "sector": ["sektor_usaha", "sektor", "sector", "industry", "industri", "business_sector",
               "jenis_usaha", "bidang_usaha", "kbli"],
    "region": ["wilayah", "region", "kab_kota", "kabupaten_kota", "kabupaten", "kota", "city",
               "provinsi", "province", "daerah", "area"],
    "scale": ["skala", "skala_usaha", "size_class", "company_size", "size_category", "ukuran", "scale", "size"],
    "umr": ["umr", "ump", "umk", "min_wage", "minimum_wage", "upah_minimum", "regional_min_wage"],
    "rm_expected": ["expected_remittance", "expected_contribution", "iuran_seharusnya", "iuran_wajib",
                    "expected_amount", "tagihan", "iuran_tagihan", "billed_amount", "expected"],
    "rm_actual": ["actual_remittance", "actual_contribution", "iuran_disetor", "iuran_dibayar", "setoran",
                  "paid_amount", "amount_paid", "remitted", "disetor", "dibayar", "actual", "paid"],
    "gt_label": ["anomaly_type", "fraud_type", "label", "injected_fraud", "fraud_label", "scenario",
                 "case_type", "jenis_fraud", "tipe", "is_fraud", "fraud"],
}

def check_privacy(df, name):
    bad = FORBIDDEN_COLS & {c.lower() for c in df.columns}
    if bad:
        raise ValueError(f"{name} berisi kolom data pribadi {bad}. Hapus dulu (requirement #6).")

def pick(df, key, override=None, required=True, exclude=()):
    if df is None:
        return None
    if override:
        if override not in df.columns:
            raise KeyError(f"Kolom '{override}' tidak ada. Kolom tersedia: {list(df.columns)}")
        return override
    lower = {c.lower(): c for c in df.columns if c not in exclude}
    for cand in CANDIDATES[key]:
        if cand in lower:
            return lower[cand]
    for cand in CANDIDATES[key]:
        if len(cand) >= 4:
            for lc, orig in lower.items():
                if cand in lc:
                    return orig
    if required:
        raise KeyError(f"Tidak bisa menebak kolom '{key}'. Isi manual di COLUMN_MAP. Kolom: {list(df.columns)}")
    return None

def load_raw(data_dir):
    raw = {}
    for name in ["headcount_timeseries", "resign_records", "payroll_timeseries", "employer_master",
                 "remittance_timeseries", "ground_truth"]:
        p = Path(data_dir) / f"{name}.csv"
        if p.exists():
            df = pd.read_csv(p)
            if name != "ground_truth":
                check_privacy(df, name)
            raw[name] = df
    return raw

def detect_columns(raw, M=COLUMN_MAP):
    c = {}
    H, R, P = raw.get("headcount_timeseries"), raw.get("resign_records"), raw.get("payroll_timeseries")
    E, RM, G = raw.get("employer_master"), raw.get("remittance_timeseries"), raw.get("ground_truth")
    if H is not None:
        c.update(hc_employer=pick(H, "employer", M["hc_employer"]), hc_period=pick(H, "period", M["hc_period"]),
                 hc_count=pick(H, "headcount", M["hc_count"]))
    if R is not None:
        c.update(rs_employer=pick(R, "employer", M["rs_employer"]), rs_period=pick(R, "period", M["rs_period"]),
                 rs_count=pick(R, "rs_count", M["rs_count"], required=False))
    if P is not None:
        c.update(pr_employer=pick(P, "employer", M["pr_employer"]), pr_period=pick(P, "period", M["pr_period"]),
                 pr_umr=pick(P, "umr", M["pr_umr"], required=False))
        used = {c["pr_employer"], c["pr_period"], c["pr_umr"]}
        c["pr_avg_wage"] = pick(P, "avg_wage", M["pr_avg_wage"], required=False, exclude=used)
        c["pr_total_wage"] = None if c["pr_avg_wage"] else pick(P, "total_wage", M["pr_total_wage"], required=False, exclude=used)
        c["pr_single_wage"] = None if (c["pr_avg_wage"] or c["pr_total_wage"]) else \
            pick(P, "single_wage", M["pr_single_wage"], required=False, exclude=used)
        used |= {c["pr_avg_wage"], c["pr_total_wage"], c["pr_single_wage"]}
        c["pr_headcount"] = pick(P, "headcount", M["pr_headcount"], required=False, exclude=used)
        if not any([c["pr_avg_wage"], c["pr_total_wage"], c["pr_single_wage"]]):
            raise KeyError(f"Kolom upah payroll tidak ketemu. Kolom: {list(P.columns)} -> isi COLUMN_MAP")
    if E is not None:
        c.update(em_employer=pick(E, "employer", M["em_employer"]), em_sector=pick(E, "sector", M["em_sector"]),
                 em_region=pick(E, "region", M["em_region"]), em_umr=pick(E, "umr", M["em_umr"], required=False))
        c["em_scale"] = pick(E, "scale", M["em_scale"], required=False,
                             exclude={c["em_employer"], c["em_sector"], c["em_region"], c["em_umr"]})
    if RM is not None:
        c.update(rm_employer=pick(RM, "employer", M["rm_employer"]), rm_period=pick(RM, "period", M["rm_period"]))
        c["rm_expected"] = pick(RM, "rm_expected", M["rm_expected"], required=False)
        c["rm_actual"] = pick(RM, "rm_actual", M["rm_actual"], exclude={c["rm_expected"]})
    if G is not None:
        c.update(gt_employer=pick(G, "employer", M["gt_employer"]), gt_label=pick(G, "gt_label", M["gt_label"]))
    return c

RAW = load_raw(DATA_DIR)
COLS = detect_columns(RAW)
print("File terbaca:", list(RAW))
print("\nMapping kolom:")
for k, v in COLS.items():
    print(f"  {k:15s} -> {v}")

## 4. Normalisasi input ke skema internal

In [ ]:
num = lambda s: pd.to_numeric(s, errors="coerce")

def to_month(s):
    s2 = s.astype(str).str.strip()
    s2 = s2.where(~s2.str.fullmatch(r"\d{6}"), s2.str[:4] + "-" + s2.str[4:])
    return pd.to_datetime(s2, errors="coerce").dt.to_period("M")

def norm_headcount(raw, c):
    H = raw.get("headcount_timeseries")
    if H is None:
        return None
    df = pd.DataFrame({"employer_id": H[c["hc_employer"]].astype(str), "period": to_month(H[c["hc_period"]]),
                       "headcount": num(H[c["hc_count"]])}).dropna()
    return df.groupby(["employer_id", "period"], as_index=False)["headcount"].sum()

def norm_resign(raw, c):
    R = raw.get("resign_records")
    if R is None:
        return None
    df = pd.DataFrame({"employer_id": R[c["rs_employer"]].astype(str), "period": to_month(R[c["rs_period"]])})
    df["n_resign"] = num(R[c["rs_count"]]).fillna(0) if c.get("rs_count") else 1
    return df.dropna(subset=["period"]).groupby(["employer_id", "period"], as_index=False)["n_resign"].sum()

def norm_payroll(raw, c, hc):
    P = raw.get("payroll_timeseries")
    if P is None:
        return None
    df = pd.DataFrame({"employer_id": P[c["pr_employer"]].astype(str), "period": to_month(P[c["pr_period"]])})
    h = num(P[c["pr_headcount"]]) if c.get("pr_headcount") else None
    df["total"], df["avg_w"], df["weight"] = np.nan, np.nan, np.nan
    if c.get("pr_avg_wage"):
        df["avg_w"] = num(P[c["pr_avg_wage"]]); df["weight"] = h if h is not None else 1.0
        df["hc"] = h if h is not None else np.nan
    elif c.get("pr_total_wage"):
        df["total"] = num(P[c["pr_total_wage"]]); df["hc"] = h if h is not None else np.nan
    else:
        df["avg_w"] = num(P[c["pr_single_wage"]]); df["weight"] = 1.0; df["hc"] = 1.0
    df["umr_p"] = num(P[c["pr_umr"]]) if c.get("pr_umr") else np.nan
    df["wx"] = df["avg_w"] * df["weight"]; df["wt"] = df["weight"].where(df["avg_w"].notna())
    df = df.dropna(subset=["period"]); df = df[df["total"].notna() | df["avg_w"].notna()]
    s_ = lambda x: x.sum(min_count=1)
    a = df.groupby(["employer_id", "period"], as_index=False).agg(
        total=("total", s_), hc=("hc", s_), wx=("wx", s_), wt=("wt", s_), umr_p=("umr_p", "median"))
    if a["hc"].isna().any() and hc is not None:
        a = a.merge(hc.rename(columns={"headcount": "hc_ts"}), on=["employer_id", "period"], how="left")
        a["hc"] = a["hc"].fillna(a["hc_ts"]); a = a.drop(columns="hc_ts")
    a["avg_wage"] = np.where(a["total"].notna(), a["total"] / a["hc"], a["wx"] / a["wt"])
    a = a[a["avg_wage"] > 0].drop(columns=["wx", "wt"]).copy()
    a["log_wage"] = np.log(a["avg_wage"])
    return a

def scale_from_headcount(n):
    out = pd.cut(n, [0, 4, 19, 99, np.inf], labels=["mikro", "kecil", "menengah", "besar"]).astype(str)
    return out.replace("nan", "tidak_diketahui")

def norm_master(raw, c, panel):
    E = raw.get("employer_master")
    if E is None:
        return None
    e = pd.DataFrame({"employer_id": E[c["em_employer"]].astype(str),
                      "sektor": E[c["em_sector"]].astype(str).str.strip(),
                      "wilayah": E[c["em_region"]].astype(str).str.strip()})
    if c.get("em_scale"):
        e["skala"] = E[c["em_scale"]].astype(str).str.strip()
    elif panel is not None:
        e["skala"] = scale_from_headcount(e["employer_id"].map(panel.groupby("employer_id")["hc"].median()))
    else:
        e["skala"] = "tidak_diketahui"
    e["umr_m"] = num(E[c["em_umr"]]) if c.get("em_umr") else np.nan
    if UMR_TABLE:
        e["umr_m"] = e["umr_m"].fillna(e["wilayah"].map(UMR_TABLE))
    return e.drop_duplicates("employer_id", keep="last").reset_index(drop=True)

def norm_remittance(raw, c, panel, hc, cfg=CFG_C):
    RM = raw.get("remittance_timeseries")
    if RM is None:
        return None, None
    df = pd.DataFrame({"employer_id": RM[c["rm_employer"]].astype(str), "period": to_month(RM[c["rm_period"]]),
                       "actual": num(RM[c["rm_actual"]])})
    if c.get("rm_expected"):
        df["expected"] = num(RM[c["rm_expected"]]); source = f"kolom '{c['rm_expected']}'"
        df = df.dropna(subset=["period"]).groupby(["employer_id", "period"], as_index=False)[["expected", "actual"]].sum()
    else:
        # hitung dari upah yang dilaporkan sendiri (payroll) x jumlah karyawan x tarif
        df = df.dropna(subset=["period"]).groupby(["employer_id", "period"], as_index=False)["actual"].sum()
        base = panel[["employer_id", "period", "avg_wage", "hc"]].copy()
        if hc is not None:
            base = base.merge(hc, on=["employer_id", "period"], how="left")
            base["hc"] = base["hc"].fillna(base["headcount"])
        base["expected"] = np.minimum(base["avg_wage"], cfg.WAGE_CAP) * base["hc"] * cfg.CONTRIB_RATE
        df = df.merge(base[["employer_id", "period", "expected"]], on=["employer_id", "period"], how="left")
        source = f"dihitung: min(upah, {cfg.WAGE_CAP:,.0f}) × jumlah karyawan × {cfg.CONTRIB_RATE:.0%}"
    return df.dropna(subset=["expected", "actual"]), source

def prepare(raw, c):
    hc = norm_headcount(raw, c)
    rs = norm_resign(raw, c)
    panel = norm_payroll(raw, c, hc)
    emp = norm_master(raw, c, panel)
    rem, rem_source = norm_remittance(raw, c, panel, hc)
    return dict(hc=hc, rs=rs, panel=panel, emp=emp, rem=rem, rem_source=rem_source)

DATA = prepare(RAW, COLS)
for k, v in DATA.items():
    if isinstance(v, pd.DataFrame):
        print(f"{k:6s}: {v['employer_id'].nunique():>5,} employer, {len(v):>7,} baris")
print("Sumber 'iuran seharusnya' Module C:", DATA["rem_source"])
if DATA["rem"] is not None:
    r = (DATA["rem"]["actual"] / DATA["rem"]["expected"]).median()
    print(f"Rasio setoran/seharusnya (median): {r:.3f}")
    if not 0.9 <= r <= 1.1:
        print("⚠ Median jauh dari 1 -> kemungkinan tarif/basis iuran berbeda (mis. hanya porsi pemberi kerja 4%). "
              "Cek CONTRIB_RATE di ConfigC.")

## 5. Module A — Registration Volatility
(Logika sama persis dengan notebook Wave 1A.)

In [ ]:
FLAGGED, NORMAL, EXPLAINED, NOT_CAND, INSUFFICIENT = \
    "FLAGGED", "NORMAL", "EXPLAINED_BY_RESIGN", "NOT_CANDIDATE", "INSUFFICIENT_DATA"
rp = lambda x: ("Rp" + f"{x:,.0f}".replace(",", ".")) if pd.notna(x) else "-"

def module_a(hc, rs, cfg=CFG_A):
    df = hc.merge(rs, on=["employer_id", "period"], how="left") if rs is not None else hc.assign(n_resign=0)
    df["n_resign"] = df["n_resign"].fillna(0)
    df = df.sort_values(["employer_id", "period"]).reset_index(drop=True)
    g = df.groupby("employer_id", sort=False)
    df["n_periods"] = g["period"].transform("count")
    df["prev_hc"] = g["headcount"].shift(1)
    df["delta"] = df["headcount"] - df["prev_hc"]
    drop = (-df["delta"]).clip(lower=0)
    df["explained"] = np.minimum(df["n_resign"], drop)
    df["delta_adj"] = df["delta"] + df["explained"]
    grp = df.groupby("employer_id", sort=False)["delta_adj"]
    W = cfg.ROLLING_WINDOW
    df["mean_delta"] = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=1).mean())
    std_hist = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=2).std())
    df["std_delta"] = np.maximum(std_hist.fillna(0), np.maximum(cfg.STD_FLOOR_ABS, cfg.STD_FLOOR_REL * df["prev_hc"]))
    df["z_raw"] = (df["delta"] - df["mean_delta"]) / df["std_delta"]
    df["z"] = (df["delta_adj"] - df["mean_delta"]) / df["std_delta"]
    df["drop_pct"] = drop / df["prev_hc"]
    df["unexplained_pct"] = (-df["delta_adj"]).clip(lower=0) / df["prev_hc"]
    base = df["mean_delta"].notna() & (df["n_periods"] >= cfg.MIN_PERIODS)
    df["flag_raw"] = base & (df["z_raw"] < -cfg.Z_THRESHOLD) & (df["drop_pct"] >= cfg.MIN_DROP_PCT)
    df["flag"] = base & (df["z"] < -cfg.Z_THRESHOLD) & (df["unexplained_pct"] >= cfg.MIN_DROP_PCT)
    df["suppressed"] = df["flag_raw"] & ~df["flag"]
    df["severity"] = np.where(base & (df["unexplained_pct"] >= cfg.MIN_DROP_PCT), (-df["z"]).clip(lower=0), 0.0)

    rows = []
    for eid, d in df.groupby("employer_id", sort=False):
        n_p = int(d["n_periods"].iloc[0])
        r = dict(employer_id=eid, n_periods=n_p)
        if n_p < cfg.MIN_PERIODS:
            rows.append({**r, "status": INSUFFICIENT, "raw": np.nan}); continue
        w = d.dropna(subset=["z"]).sort_values(["flag", "severity", "z"], ascending=[False, False, True]).iloc[0]
        if d["flag"].any():
            st = FLAGGED
        elif d["suppressed"].any():
            st = EXPLAINED; w = d[d["suppressed"]].sort_values("z_raw").iloc[0]
        else:
            st = NORMAL
        rows.append({**r, "status": st, "raw": float(d["severity"].max()),
                     "n_flagged_periods": int(d["flag"].sum()), "worst_period": str(w["period"]),
                     "hc_before": w["prev_hc"], "hc_after": w["headcount"], "drop": max(-w["delta"], 0),
                     "drop_pct": w["drop_pct"], "resign_recorded": w["n_resign"], "z_score": float(w["z"])})
    out = pd.DataFrame(rows)

    def reason(r):
        if r.status == INSUFFICIENT:
            return f"Riwayat jumlah karyawan baru {r.n_periods} bulan (< {cfg.MIN_PERIODS}); belum bisa dinilai."
        if r.status == FLAGGED:
            rs_txt = (f"hanya {int(r.resign_recorded)} orang tercatat resign" if r.resign_recorded > 0
                      else "tidak ada catatan resign yang cocok")
            return (f"Jumlah karyawan turun {r.drop_pct:.0%} ({int(r.hc_before)}→{int(r.hc_after)} orang) pada "
                    f"{r.worst_period}, {rs_txt}.")
        if r.status == EXPLAINED:
            return (f"Jumlah karyawan turun {int(r.drop)} orang pada {r.worst_period}, sesuai catatan resign resmi "
                    f"({int(r.resign_recorded)} orang) — wajar.")
        return "Pergerakan jumlah karyawan dalam batas wajar."
    out["reason"] = [reason(r) for r in out.itertuples()]
    return out, df

if DATA["hc"] is not None:
    RES_A, DET_A = module_a(DATA["hc"], DATA["rs"])
    print(RES_A["status"].value_counts().to_string())
else:
    RES_A, DET_A = None, None; print("⚠ headcount_timeseries tidak ada -> Module A dilewati")

## 6. Module B — Peer-Group Wage Benchmarking
(Logika sama dengan notebook Wave 1B.)

In [ ]:
COHORT_LEVELS = [("L0", ["sektor", "wilayah", "skala"]), ("L1", ["sektor", "wilayah"]),
                 ("L2", ["sektor", "skala"]), ("L3", ["sektor"]), ("L4", [])]
LEVEL_NOTE = {"L0": "", "L1": " (skala direlaksasi)", "L2": " (wilayah direlaksasi)",
              "L3": " (hanya sektor)", "L4": " (semua employer)"}

def assign_cohorts(emp, cfg):
    e = emp.copy(); e["cohort_level"] = None; e["cohort_size"] = np.nan
    for lvl, keys in COHORT_LEVELS:
        size = e.groupby(keys)["employer_id"].transform("nunique") if keys else pd.Series(len(e), index=e.index)
        m = e["cohort_level"].isna() & (size >= cfg.MIN_COHORT_SIZE)
        e.loc[m, "cohort_level"] = lvl; e.loc[m, "cohort_size"] = size[m]
    m = e["cohort_level"].isna(); e.loc[m, "cohort_level"] = "L4"; e.loc[m, "cohort_size"] = len(e)
    kd = dict(COHORT_LEVELS)
    e["cohort_key"] = [", ".join(f"{k} {r[k]}" for k in kd[l]) or "semua perusahaan"
                       for l, (_, r) in zip(e["cohort_level"], e.iterrows())]
    return e

def module_b(panel, emp, cfg=CFG_B):
    emp = emp[emp["employer_id"].isin(panel["employer_id"])]
    p = panel.merge(emp, on="employer_id", how="inner")
    p["umr"] = p["umr_p"].fillna(p["umr_m"])
    has_umr = p["umr"].notna().any()
    emp_c = assign_cohorts(emp, cfg)
    p = p.merge(emp_c[["employer_id", "cohort_level", "cohort_size", "cohort_key"]], on="employer_id")
    parts = []
    for lvl, keys in COHORT_LEVELS:
        sub = p[p["cohort_level"] == lvl]
        if sub.empty:
            continue
        gk = keys + ["period"]
        st = (p.groupby(gk)["log_wage"].agg(c_median="median", c_q1=lambda s: s.quantile(.25),
                                            c_q3=lambda s: s.quantile(.75), c_n="count").reset_index())
        parts.append(sub.merge(st, on=gk, how="left"))
    d = pd.concat(parts, ignore_index=True)
    d["c_iqr"] = np.maximum(d["c_q3"] - d["c_q1"], cfg.IQR_FLOOR)
    d["z"] = np.where(d["c_n"] >= cfg.MIN_COHORT_SIZE, (d["log_wage"] - d["c_median"]) / d["c_iqr"], np.nan)
    d["pct_vs_median"] = np.exp(d["log_wage"] - d["c_median"]) - 1
    d["cohort_median_wage"] = np.exp(d["c_median"])
    d["below"] = (d["z"] < -cfg.Z_THRESHOLD).where(d["z"].notna())

    e = d.groupby("employer_id").agg(
        n_periods=("period", "nunique"), n_valid=("z", "count"), n_below=("below", "sum"), z_median=("z", "median"),
        pct_vs_median=("pct_vs_median", "median"), wage_median=("avg_wage", "median"),
        cohort_median_wage=("cohort_median_wage", "median"), umr=("umr", "median"),
        cohort_key=("cohort_key", "first"), cohort_level=("cohort_level", "first"),
        cohort_size=("cohort_size", "first")).reset_index()
    e["frac_below"] = e["n_below"] / e["n_valid"].replace(0, np.nan)
    if has_umr and cfg.UMR_PREFILTER_MULT is not None:
        e["candidate"] = e["umr"].isna() | (e["wage_median"] <= cfg.UMR_PREFILTER_MULT * e["umr"])
    else:
        e["candidate"] = True
    enough = e["n_periods"] >= cfg.MIN_PERIODS
    flag = enough & e["candidate"] & (e["z_median"] < -cfg.Z_THRESHOLD) & (e["frac_below"] >= cfg.MIN_PERSIST)
    e["status"] = np.select([~enough, ~e["candidate"], flag], [INSUFFICIENT, NOT_CAND, FLAGGED], NORMAL)
    e["raw"] = np.where(e["status"].isin([FLAGGED, NORMAL]), (-e["z_median"]).clip(lower=0).fillna(0), 0.0)
    e.loc[e["status"] == INSUFFICIENT, "raw"] = np.nan

    def reason(r):
        cohort = f"{r.cohort_key}{LEVEL_NOTE[r.cohort_level]}, {int(r.cohort_size)} perusahaan"
        if r.status == INSUFFICIENT:
            return f"Data upah baru {r.n_periods} bulan (< {cfg.MIN_PERIODS}); belum bisa dinilai."
        if r.status == NOT_CAND:
            return f"Upah rata-rata {rp(r.wage_median)}/orang di atas {cfg.UMR_PREFILTER_MULT}× UMR; tidak masuk kandidat."
        if r.status == FLAGGED:
            umr = (" Upah juga di bawah UMR, tetapi itu sendiri bukan dasar flag (urusan Disnaker)."
                   if pd.notna(r.umr) and r.wage_median < r.umr else "")
            return (f"Upah dilaporkan {rp(r.wage_median)}/orang, {abs(r.pct_vs_median):.0%} di bawah median perusahaan "
                    f"sejenis ({rp(r.cohort_median_wage)}; {cohort}), konsisten di {int(r.n_below)}/{int(r.n_valid)} bulan.{umr}")
        arah = "di bawah" if r.pct_vs_median < 0 else "di atas"
        return f"Upah sejalan dengan perusahaan sejenis ({abs(r.pct_vs_median):.0%} {arah} median; {cohort})."
    e["reason"] = [reason(r) for r in e.itertuples()]
    return e, d

if DATA["panel"] is not None and DATA["emp"] is not None:
    RES_B, DET_B = module_b(DATA["panel"], DATA["emp"])
    print(RES_B["status"].value_counts().to_string())
else:
    RES_B, DET_B = None, None; print("⚠ payroll/employer_master tidak lengkap -> Module B dilewati")

## 7. Module C — Contribution Reconciliation
(Kode **identik** dengan notebook Wave 1C: tolerance band absolut + persentase, telat bayar yang dilunasi bulan berikutnya tidak dihitung, flag hanya kalau kurang setor **≥ 2 bulan berturut-turut**.)

In [ ]:
rp = lambda x: ("Rp" + f"{x:,.0f}".replace(",", ".")) if pd.notna(x) else "-"

ONE_OFF = "ONE_OFF_GAP"

def module_c(rem, cfg=None):
    """rem: employer_id, period, expected, actual  ->  (tabel per employer, detail per periode)"""
    cfg = cfg or CFG_C
    d = rem.sort_values(["employer_id", "period"]).reset_index(drop=True).copy()
    d["gap"] = d["expected"] - d["actual"]                        # selisih = seharusnya - disetor
    d["gap_pct"] = np.where(d["expected"] > 0, d["gap"] / d["expected"], np.nan)
    d["over_tol"] = (d["gap"] > cfg.TOL_ABS) & (d["gap_pct"] > cfg.TOL_PCT)   # tolerance band
    d["pidx"] = d["period"].dt.year * 12 + d["period"].dt.month
    g = d.groupby("employer_id", sort=False)
    nxt_is_next_month = (g["pidx"].shift(-1) - d["pidx"]) == 1
    nxt_surplus = -g["gap"].shift(-1)                              # kelebihan bayar bulan berikutnya
    d["late_paid"] = (cfg.ALLOW_CATCHUP & d["over_tol"] & nxt_is_next_month
                      & (nxt_surplus >= d["gap"] * (1 - cfg.TOL_PCT)))
    d["is_gap"] = d["over_tol"] & ~d["late_paid"]

    # panjang run berturut-turut
    run_len = np.zeros(len(d), dtype=int)
    prev_e, prev_idx, prev_gap, cur = None, None, False, 0
    for i, (e, p, gp) in enumerate(zip(d["employer_id"], d["pidx"], d["is_gap"])):
        if gp:
            cur = cur + 1 if (e == prev_e and prev_gap and p == prev_idx + 1) else 1
        else:
            cur = 0
        run_len[i] = cur
        prev_e, prev_idx, prev_gap = e, p, gp
    d["run_len"] = run_len

    rows = []
    for eid, x in d.groupby("employer_id", sort=False):
        n_p, gaps = len(x), x[x["is_gap"]]
        longest = int(x["run_len"].max())
        r = dict(employer_id=eid, n_periods=n_p, n_gap_periods=len(gaps), longest_run=longest,
                 n_late_paid=int(x["late_paid"].sum()),
                 total_expected=float(x["expected"].sum()), total_actual=float(x["actual"].sum()),
                 total_shortfall=float(gaps["gap"].sum()),
                 median_gap_pct=float(gaps["gap_pct"].median()) if len(gaps) else 0.0,
                 run_start=None, run_end=None, first_gap_period=str(gaps["period"].min()) if len(gaps) else None)
        r["cum_shortfall_pct"] = r["total_shortfall"] / r["total_expected"] if r["total_expected"] > 0 else np.nan
        if longest > 0:
            end_i = x["run_len"].idxmax()
            r["run_end"] = str(x.loc[end_i, "period"])
            r["run_start"] = str(x.loc[end_i - longest + 1, "period"])
        if n_p < cfg.MIN_PERIODS:
            r.update(status=INSUFFICIENT, raw=np.nan)
        elif longest >= cfg.MIN_CONSECUTIVE:
            r.update(status=FLAGGED, raw=r["median_gap_pct"])
        elif len(gaps):
            r.update(status=ONE_OFF, raw=float(gaps["gap_pct"].max()))
        else:
            r.update(status=NORMAL, raw=0.0)
        rows.append(r)
    e = pd.DataFrame(rows)

    def reason(r):
        late = (f" {r.n_late_paid} kali telat setor tapi dilunasi bulan berikutnya (dianggap wajar)."
                if r.n_late_paid else "")
        if r.status == INSUFFICIENT:
            return f"Data setoran baru {r.n_periods} bulan; belum bisa dinilai."
        if r.status == FLAGGED:
            extra = (f" Ada {r.n_gap_periods - r.longest_run} bulan lain yang juga kurang setor."
                     if r.n_gap_periods > r.longest_run else "")
            return (f"Setoran iuran rata-rata {r.median_gap_pct:.0%} di bawah yang seharusnya selama "
                    f"{r.longest_run} bulan berturut-turut ({r.run_start} s/d {r.run_end}); "
                    f"total kekurangan {rp(r.total_shortfall)}.{extra}{late}")
        if r.status == ONE_OFF:
            return (f"Kurang setor pada {r.first_gap_period}, tetapi tidak berulang berturut-turut — "
                    f"dianggap wajar.{late}")
        return (f"Setoran sesuai dengan yang seharusnya (selisih dalam toleransi "
                f"{cfg.TOL_PCT:.0%} / {rp(cfg.TOL_ABS)}).{late}")
    e["reason"] = [reason(r) for r in e.itertuples()]
    return e, d

if DATA["rem"] is not None:
    RES_C, DET_C = module_c(DATA["rem"])
    print(RES_C["status"].value_counts().to_string())
else:
    RES_C, DET_C = None, None; print("⚠ remittance_timeseries tidak ada -> Module C dilewati")

## 8. Normalisasi, composite score, band & explainability
**Normalisasi (piecewise linear, berbasis threshold):**
- skor mentah `0` → `0.0`; tepat di **threshold** → `0.5`; ≥ **batas atas** (`threshold × CAP`) → `1.0`
- modul yang tidak FLAGGED dibatasi maks `0.49`; modul FLAGGED minimal `0.5`

| Modul | Skor mentah | Threshold (→0.5) | Batas atas (→1.0) |
|---|---|---|---|
| A | z-score penurunan headcount yang tidak terjelaskan | `Z_THRESHOLD` A | `Z_THRESHOLD × CAP_MULT` |
| B | berapa IQR di bawah median cohort | `Z_THRESHOLD` B | `Z_THRESHOLD × CAP_MULT` |
| C | median % kekurangan setor (bulan yang kurang setor) | `TOL_PCT` | `CAP_PCT` |

**Band:** `Tinggi` = ≥2 modul flag, **atau** 1 modul flag dengan skor ≥ `STRONG_SCORE` (bukti sangat kuat) · `Sedang` = 1 modul flag · `Rendah` = tidak ada flag · `Belum bisa dinilai` = tidak ada modul yang bisa menilai.

**Urutan dalam band:** jumlah modul flag → skor modul tertinggi (`max_module_score`) → `composite_score`.

In [ ]:
BAND_ORDER = {"Tinggi": 0, "Sedang": 1, "Rendah": 2, "Belum bisa dinilai": 3}
BAND_CODE = {"Tinggi": "HIGH", "Sedang": "MEDIUM", "Rendah": "LOW", "Belum bisa dinilai": "INSUFFICIENT"}

def normalize(raw, status, thr, cap):
    """Normalisasi berbasis threshold: 0 -> 0.0, threshold -> 0.5, cap -> 1.0.
    Modul yang tidak FLAGGED dibatasi <= 0.49, yang FLAGGED >= 0.5."""
    raw = pd.Series(raw, dtype=float).clip(lower=0)
    below = 0.5 * raw / thr
    above = 0.5 + 0.5 * (raw - thr) / (cap - thr)
    n = pd.Series(np.where(raw < thr, below, above), index=raw.index).clip(0, 1)
    flagged = pd.Series(status, index=raw.index).eq(FLAGGED)
    n = pd.Series(np.where(flagged, np.maximum(n, 0.5), np.minimum(n, 0.49)), index=raw.index)
    return n.where(raw.notna()).round(3)

NORM_PARAMS = {
    "A": (CFG_A.Z_THRESHOLD, CFG_A.Z_THRESHOLD * CFG_A.CAP_MULT),
    "B": (CFG_B.Z_THRESHOLD, CFG_B.Z_THRESHOLD * CFG_B.CAP_MULT),
    "C": (CFG_C.TOL_PCT, CFG_C.CAP_PCT),
}

def short_text(m, r):
    # kalimat ringkas untuk UI (satu baris per modul)
    if m == "A":
        rs_txt = "tidak ada resign record yang cocok" if r["resign_recorded"] == 0 else f"hanya {int(r['resign_recorded'])} resign tercatat"
        return f"Headcount turun {r['drop_pct']:.0%} ({int(r['hc_before'])}→{int(r['hc_after'])}) pada {r['worst_period']}, {rs_txt}"
    if m == "B":
        return f"Upah {abs(r['pct_vs_median']):.0%} di bawah median cohort {r['cohort_key']}"
    return f"Setoran {r['median_gap_pct']:.0%} di bawah seharusnya selama {int(r['longest_run'])} bulan berturut-turut ({r['run_start']} s/d {r['run_end']})"

METRIC_COLS = {
    "A": ["worst_period", "hc_before", "hc_after", "drop", "drop_pct", "resign_recorded", "z_score", "n_flagged_periods", "n_periods"],
    "B": ["wage_median", "cohort_median_wage", "pct_vs_median", "z_median", "frac_below", "cohort_key", "cohort_level",
          "cohort_size", "umr", "candidate", "n_periods"],
    "C": ["longest_run", "run_start", "run_end", "n_gap_periods", "n_late_paid", "n_periods", "median_gap_pct",
          "cum_shortfall_pct", "total_shortfall"],
}

def composite(results, emp, cfg=CFG_X):
    ids = set()
    for r in results.values():
        if r is not None:
            ids |= set(r["employer_id"])
    base = pd.DataFrame({"employer_id": sorted(ids)})
    if emp is not None:
        base = base.merge(emp[["employer_id", "sektor", "wilayah", "skala"]], on="employer_id", how="left")
    mods = {}
    for m, r in results.items():
        if r is None:
            continue
        thr, cap = NORM_PARAMS[m]
        t = r.copy(); t["norm"] = normalize(t["raw"], t["status"], thr, cap)
        mods[m] = t.set_index("employer_id")
        base[f"status_{m}"] = base["employer_id"].map(t.set_index("employer_id")["status"]).fillna("NO_DATA")
        base[f"score_{m}"] = base["employer_id"].map(t.set_index("employer_id")["norm"])
    sc = [c for c in base.columns if c.startswith("score_")]
    base["coverage"] = base[sc].notna().sum(axis=1)
    base["composite_score"] = (base[sc].mean(axis=1) if cfg.METHOD == "mean" else base[sc].max(axis=1)).round(3)
    base["max_module_score"] = base[sc].max(axis=1)
    base["n_flags"] = sum(base[f"status_{m}"].eq(FLAGGED).astype(int) for m in mods)
    strong = (base["n_flags"] >= 2) | ((base["n_flags"] == 1) & (base["max_module_score"] >= cfg.STRONG_SCORE))
    base["band"] = np.select([base["coverage"] == 0, strong, base["n_flags"] == 1],
                             ["Belum bisa dinilai", "Tinggi", "Sedang"], "Rendah")
    base = (base.assign(_o=base["band"].map(BAND_ORDER))
                .sort_values(["_o", "n_flags", "max_module_score", "composite_score"],
                             ascending=[True, False, False, False], na_position="last")
                .drop(columns="_o").reset_index(drop=True))
    base.insert(0, "rank", range(1, len(base) + 1))
    return base, mods

def explain(row, mods):
    # payload penjelasan per employer: kontribusi tiap modul + nilai mentah pendukung
    avail = {m: mods[m].loc[row.employer_id] for m in mods if row.employer_id in mods[m].index
             and pd.notna(mods[m].loc[row.employer_id, "norm"])}
    total = sum(float(r["norm"]) for r in avail.values())
    drivers, notes = [], []
    for m in ["A", "B", "C"]:
        if m not in mods:
            notes.append(f"Module {m} ({MODULE_LABEL[m]}) tidak dijalankan: data tidak tersedia.")
            continue
        if row.employer_id not in mods[m].index:
            notes.append(f"Module {m} ({MODULE_LABEL[m]}): tidak ada data untuk perusahaan ini.")
            continue
        r = mods[m].loc[row.employer_id]
        if r["status"] == INSUFFICIENT:
            notes.append(f"Module {m} ({MODULE_LABEL[m]}): {r['reason']}")
            continue
        drivers.append(dict(
            module=m, label=MODULE_LABEL[m], status=r["status"], flagged=bool(r["status"] == FLAGGED),
            score=float(r["norm"]), contribution_pct=round(100 * float(r["norm"]) / total, 1) if total > 0 else 0.0,
            short_text=short_text(m, r) if r["status"] == FLAGGED else None, text=r["reason"],
            metrics={k: r[k] for k in METRIC_COLS[m] if k in r.index}))
    drivers.sort(key=lambda x: x["score"], reverse=True)
    flagged = [d for d in drivers if d["flagged"]]
    if row.band == "Belum bisa dinilai":
        summary = "Belum bisa dinilai: data belum cukup di semua modul."
    elif flagged:
        summary = f"Prioritas {row.band.lower()} — {len(flagged)} dari {row.coverage} modul menunjukkan indikasi: " + \
                  "; ".join(d["short_text"] for d in flagged) + "."
    else:
        summary = f"Tidak ada indikasi di {row.coverage} modul yang dinilai."
    if 0 < row.coverage < len(mods):
        notes.append(f"Skor gabungan dihitung dari {row.coverage} dari {len(mods)} modul yang tersedia.")
    return dict(summary=summary, drivers=drivers, notes=notes)

COMP, MODS = composite({"A": RES_A, "B": RES_B, "C": RES_C}, DATA["emp"])
COMP["explanation"] = [explain(r, MODS) for r in COMP.itertuples()]
print(COMP["band"].value_counts().reindex(BAND_ORDER).fillna(0).astype(int).to_string())
COMP.head(10).drop(columns="explanation")

In [ ]:
# Contoh payload penjelasan untuk perusahaan ranking #1
def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if pd.isna(o) else round(float(o), 4)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, pd.Period):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o

print(json.dumps(_jsonable(COMP.iloc[0]["explanation"]), indent=2, ensure_ascii=False))

## 9. Evaluasi end-to-end vs `ground_truth.csv`
- **Per modul**: recall & false positive rate terhadap label jenisnya masing-masing.
- **Composite**: seberapa banyak kasus curang (jenis apa pun) ada di **puncak daftar**. `precision@K` = dari K perusahaan teratas, berapa persen yang memang curang. Ini metrik yang paling relevan untuk tim pemeriksa (requirement #4: kerja terarah).

In [ ]:
def build_truth(raw, c):
    G = raw.get("ground_truth")
    if G is None:
        return None
    lab = G[c["gt_label"]].astype(str).str.upper().str.strip()
    t = pd.DataFrame({"employer_id": G[c["gt_employer"]].astype(str), "label": lab})
    for m, pats in POSITIVE_LABELS.items():
        t[f"is_{m}"] = lab.str.contains("|".join(pats), na=False)
    t["is_any"] = ~lab.isin(NEGATIVE_LABELS)
    return t.groupby("employer_id", as_index=False).agg(
        label=("label", lambda s: ",".join(sorted(set(s)))), **{k: (k, "any") for k in ["is_A", "is_B", "is_C", "is_any"]})

def prf(pred, actual):
    tp, fp = int((pred & actual).sum()), int((pred & ~actual).sum())
    fn, tn = int((~pred & actual).sum()), int((~pred & ~actual).sum())
    return dict(tp=tp, fp=fp, fn=fn, tn=tn,
                precision=round(tp / (tp + fp), 3) if tp + fp else np.nan,
                recall=round(tp / (tp + fn), 3) if tp + fn else np.nan,
                fpr=round(fp / (fp + tn), 3) if fp + tn else np.nan)

def evaluate(comp, truth):
    m = comp.merge(truth, on="employer_id", how="left")
    for c in ["is_A", "is_B", "is_C", "is_any"]:
        m[c] = m[c].fillna(False).astype(bool)
    rows = []
    for mod in ["A", "B", "C"]:
        if f"status_{mod}" not in m:
            continue
        ev = m[~m[f"status_{mod}"].isin([INSUFFICIENT, "NO_DATA"])]
        rows.append(dict(level=f"Module {mod}", **prf(ev[f"status_{mod}"].eq(FLAGGED), ev[f"is_{mod}"])))
    ev = m[m["band"] != "Belum bisa dinilai"]
    rows.append(dict(level="Composite (band Tinggi/Sedang)", **prf(ev["band"].isin(["Tinggi", "Sedang"]), ev["is_any"])))
    n_pos = int(m["is_any"].sum())
    at_k = {f"precision@{k}": round(float(m.head(k)["is_any"].mean()), 3) for k in (10, 25, 50, n_pos) if 0 < k <= len(m)}
    return pd.DataFrame(rows), at_k, m

TRUTH = build_truth(RAW, COLS)
if TRUTH is not None:
    print("Label ground truth:", TRUTH["label"].value_counts().to_dict(), "\n")
    EVAL_TBL, AT_K, EVAL_M = evaluate(COMP, TRUTH)
    display(EVAL_TBL) if "display" in globals() else print(EVAL_TBL.to_string(index=False))
    print("\nKualitas urutan daftar prioritas:", AT_K)

    miss = EVAL_M[EVAL_M["is_any"] & ~EVAL_M["band"].isin(["Tinggi", "Sedang"])]
    if len(miss):
        print(f"\nKasus curang yang tidak masuk band Tinggi/Sedang ({len(miss)}):")
        print(miss[["rank", "employer_id", "label", "band", "composite_score"] +
                   [c for c in miss.columns if c.startswith("status_")]].head(15).to_string(index=False))
else:
    print("ground_truth.csv tidak ada — evaluasi dilewati")

## 10. Cross-check vs output Wave 1 (A, B, C)
Membandingkan status per employer di pipeline ini dengan file `module_a/b/c_scores.csv` hasil notebook Wave 1.
- **100% cocok** → logika & konfigurasi sama.
- **Ada yang beda** → cek apakah `ConfigA/B/C`, `COLUMN_MAP`, `UMR_TABLE`, atau label di sini sudah sama dengan notebook modulnya (misal threshold hasil tuning baru ditulis di satu tempat saja).
- File Wave 1 yang tidak ditemukan dilewati (tidak error).

In [ ]:
def find_latest(filename):
    root = Path(os.environ["HK_WAVE1_DIR"]) if os.environ.get("HK_WAVE1_DIR") else \
           Path("/content/drive/MyDrive") if IN_COLAB else Path(".")
    hits = [Path(p) for p in glob.glob(str(root / "**" / filename), recursive=True)]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

XCHECK = []
for m, res in {"A": RES_A, "B": RES_B, "C": RES_C}.items():
    p = find_latest(f"module_{m.lower()}_scores.csv")
    if p is None or res is None:
        print(f"(—) Module {m}: file Wave 1 tidak ditemukan / modul tidak jalan, dilewati"); continue
    w1 = pd.read_csv(p)[["employer_id", "status"]].astype({"employer_id": str})
    cmp_ = res[["employer_id", "status"]].merge(w1, on="employer_id", how="outer", suffixes=("_wave2", "_wave1"))
    same = cmp_["status_wave2"] == cmp_["status_wave1"]
    fl2, fl1 = set(res.loc[res.status == FLAGGED, "employer_id"]), set(w1.loc[w1.status == FLAGGED, "employer_id"])
    XCHECK.append(dict(modul=m, file_wave1=str(p), n_employer=len(cmp_), status_cocok_pct=round(100 * float(same.mean()), 1),
                       flagged_wave2=len(fl2), flagged_wave1=len(fl1), flagged_sama=len(fl2 & fl1)))
    if not same.all():
        print(f"⚠ Module {m}: {int((~same).sum())} employer beda status. Contoh:")
        print(cmp_[~same].head(8).to_string(index=False))
XCHECK = pd.DataFrame(XCHECK)
if len(XCHECK):
    display(XCHECK) if "display" in globals() else print(XCHECK.to_string(index=False))
    if (XCHECK["status_cocok_pct"] == 100).all():
        print("\n✅ Hasil Wave 2 identik dengan output Wave 1 untuk semua modul yang dicek")

## 11. Export → `output_scores.json` (+ CSV)
Struktur JSON:
```json
{
  "meta": { "generated_at": "...", "method": "...", "config": {...}, "counts": {...}, "disclaimer": "..." },
  "companies": [
    {
      "id": "EMP-0002", "rank": 1, "sektor": "...", "wilayah": "...", "skala": "...",
      "composite_score": 0.41, "max_module_score": 0.93, "band": "Tinggi", "band_code": "HIGH",
      "coverage": 3, "modules_flagged": ["C"],
      "scores":  { "A": 0.12, "B": 0.20, "C": 0.93 },
      "status":  { "A": "NORMAL", "B": "NORMAL", "C": "FLAGGED" },
      "recommendation": "Perlu dicek",
      "explanation": { "summary": "...", "drivers": [ {"module": "C", "label": "...", "score": 0.93,
                        "contribution_pct": 74.4, "short_text": "...", "text": "...", "metrics": {...}} ],
                       "notes": ["..."] }
    }
  ]
}
```
`band_code` (HIGH/MEDIUM/LOW/INSUFFICIENT) disediakan supaya gampang dipetakan ke warna/label di UI.

In [ ]:
REC = {"Tinggi": "Perlu dicek segera", "Sedang": "Perlu dicek", "Rendah": "Tidak ada indikasi",
       "Belum bisa dinilai": "Belum bisa dinilai — pantau"}

def to_company(r, mods_run):
    return _jsonable(dict(
        id=r.employer_id, rank=r.rank,
        sektor=getattr(r, "sektor", None), wilayah=getattr(r, "wilayah", None), skala=getattr(r, "skala", None),
        composite_score=r.composite_score, max_module_score=r.max_module_score,
        band=r.band, band_code=BAND_CODE[r.band],
        coverage=r.coverage, modules_flagged=[m for m in mods_run if getattr(r, f"status_{m}") == FLAGGED],
        scores={m: getattr(r, f"score_{m}") for m in mods_run},
        status={m: getattr(r, f"status_{m}") for m in mods_run},
        recommendation=REC[r.band], explanation=r.explanation))

def export(comp, mods, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    mods_run = list(mods)
    payload = {
        "meta": _jsonable(dict(
            generated_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
            data_source=str(DATA_DIR), modules_run=mods_run,
            method=f"composite = {CFG_X.METHOD} dari skor modul ternormalisasi (threshold-based, 0.5 = batas flag)",
            config=dict(A=asdict(CFG_A), B=asdict(CFG_B), C=asdict(CFG_C), composite=asdict(CFG_X)),
            remittance_expected_source=DATA["rem_source"],
            counts=dict(total=len(comp), **comp["band"].value_counts().to_dict()),
            disclaimer="Data simulasi. Daftar ini adalah rekomendasi prioritas pemeriksaan, bukan vonis. "
                       "Keputusan akhir ada di tim pemeriksa BPJS.")),
        "companies": [to_company(r, mods_run) for r in comp.itertuples()],
    }
    jpath = out_dir / "output_scores.json"
    with open(jpath, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    flat = comp.drop(columns="explanation").assign(
        summary=[e["summary"] for e in comp["explanation"]])
    flat.to_csv(out_dir / "output_scores.csv", index=False)
    return jpath, payload

JPATH, PAYLOAD = export(COMP, MODS, OUT_DIR)
print("Tersimpan:")
for f in sorted(Path(OUT_DIR).glob("output_scores.*")):
    print(f"  ✓ {f}  ({f.stat().st_size/1024:.0f} KB)")
print("\nContoh 1 perusahaan:")
print(json.dumps(PAYLOAD["companies"][0], indent=2, ensure_ascii=False)[:2500])

## 12. Validasi output (Definition of Done)
Cek otomatis bahwa JSON valid, lengkap, dan konsisten sebelum dipakai prototype.

In [ ]:
with open(JPATH, encoding="utf-8") as f:
    J = json.load(f)
cs = J["companies"]
checks = {
    "JSON bisa dibaca ulang": True,
    "semua employer ada": len(cs) == len(COMP),
    "id unik": len({c["id"] for c in cs}) == len(cs),
    "urut dari paling berisiko (band dulu)": all(
        BAND_ORDER[a["band"]] <= BAND_ORDER[b["band"]] for a, b in zip(cs, cs[1:])),
    "skor dalam 0–1 atau null": all(c["composite_score"] is None or 0 <= c["composite_score"] <= 1 for c in cs),
    "tiap employer yang flag punya penjelasan": all(
        c["explanation"]["summary"] and any(d["flagged"] for d in c["explanation"]["drivers"])
        for c in cs if c["modules_flagged"]),
    "tidak ada kolom data pribadi": not any(k in json.dumps(J).lower() for k in ['"nik"', '"nama_karyawan"']),
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
assert all(checks.values()), "Ada cek yang gagal — lihat di atas"
print(f"\nDefinition of done Wave 2 terpenuhi: {JPATH}")